In [2]:
import pandas as pd
import numpy as np
import json

# Extract data from RegulonDB

In [2]:
from pymongo import MongoClient

# Configura la conexión a MongoDB
client = MongoClient('mongodb://localhost:27017/')  # Cambia la URL según tu configuración
db = client['regulondbht']  # Reemplaza con el nombre de tu base de datos
collection = db['geneExpression']  # Reemplaza con el nombre de tu colección


In [3]:

# Define el pipeline de agregación
pipeline = [
    {
        "$group": {
            "_id": "$temporalId",            
            "bnumber_fpkm_tpm": {
                "$push": {
                    "bnumber": "$gene.bnumber",
                    "name": "$gene.name",
                    "gene_id": "$gene._id",
                    "values": {
                        "count": "$count",
                        "fpkm": "$fpkm",
                        "tpm": "$tpm"
                    }
                }
            }
        }
    }
]


In [18]:

# Ejecuta la agregación
results = collection.aggregate(pipeline)


In [19]:
with open('RegulonDB/RegulonDB_v12/RNAseq/geneExpression.json', 'w') as file:
    for result in results:
        json.dump(result, file, indent=4)
        file.write(',\n')  # Añade una nueva línea después de cada documento para separar los JSON


In [ ]:

with open('RegulonDB/RegulonDB_v12/RNAseq/geneExpression.json', 'r+') as file:
    file.seek(0, 2)  # Mueve el cursor al final del archivo
    file.seek(file.tell() - 2, 0)  # Retrocede dos caracteres (la coma y nueva línea)
    file.truncate()  # Trunca el archivo en esta posición
    file.write('\n]')  # Añade la nueva línea final y cierra el array JSON

In [3]:
file_path = "RegulonDB/RegulonDB_v12/RNAseq/geneExpression.json"

with open(file_path, 'r') as file:
  data = json.load(file)

In [9]:
total_length = len(data)
segment_size = total_length // 4

# Dividir los datos en cuatro partes
part1 = data[:segment_size]
part2 = data[segment_size:2*segment_size]
part3 = data[2*segment_size:3*segment_size]
part4 = data[3*segment_size:]

# Guardar cada parte en un archivo separado
with open('RegulonDB/RegulonDB_v12/RNAseq/geneExpression_part1.json', 'w') as file:
    json.dump(part1, file, indent=4)

with open('RegulonDB/RegulonDB_v12/RNAseq/geneExpression_part2.json', 'w') as file:
    json.dump(part2, file, indent=4)

with open('RegulonDB/RegulonDB_v12/RNAseq/geneExpression_part3.json', 'w') as file:
    json.dump(part3, file, indent=4)

with open('RegulonDB/RegulonDB_v12/RNAseq/geneExpression_part4.json', 'w') as file:
    json.dump(part4, file, indent=4)

# Parse files

In [3]:
file_path = "RegulonDB/RegulonDB_v12/RNAseq/geneExpression_part1.json"

with open(file_path, 'r') as file:
  data = json.load(file)

In [ ]:
df_count = pd.DataFrame()
df_fpkm = pd.DataFrame()
df_tpm = pd.DataFrame()

In [ ]:
i=0
for j in range(1, 5):
    file_path = f"/content/drive/MyDrive/geneExpression_part{j}.json"
    with open(file_path, 'r') as file:
        data = json.load(file)
    for dataset in data:
        print(i)
        dataset_id = dataset['_id']
        df_count.loc[i, "dataset id"] = dataset_id
        df_fpkm.loc[i, "dataset id"] = dataset_id
        df_tpm.loc[i, "dataset id"] = dataset_id
        for gene in dataset["bnumber_fpkm_tpm"]:
            try:
                gene_name = gene['name']
            except:
                continue
            try:
                gene_count = gene['values']['count']
                df_count.loc[i, f"{gene_name}"] = gene_count
            except:
                df_count.loc[i, f"{gene_name}"] = np.NaN
            try:
                gene_fpkm = gene['values']['fpkm']
                df_fpkm.loc[i, f"{gene_name}"] = gene_fpkm
            except:
                df_fpkm.loc[i, f"{gene_name}"] = np.NaN
            try:
                gene_tpm = gene['values']['tpm']
                df_tpm.loc[i, f"{gene_name}"] = gene_tpm
            except:
                df_tpm.loc[i, f"{gene_name}"] = np.NaN
        i = i+1
    df_count.to_csv(f"/content/drive/MyDrive/count_{j}.csv")
    df_fpkm.to_csv(f"/content/drive/MyDrive/fpkm_{j}.csv")
    df_tpm.to_csv(f"/content/drive/MyDrive/tpm_{j}.csv")
        
        

# Load dataframes

In [4]:
df_count = pd.read_csv("RegulonDB\RegulonDB_v12\RNAseq\count_4.csv")

In [6]:
df_fpkm = pd.read_csv("RegulonDB\\RegulonDB_v12\\RNAseq\\fpkm_4.csv")

In [39]:
df_tpm = pd.read_csv("RegulonDB\\RegulonDB_v12\\RNAseq\\tpm_4.csv")

In [14]:
df_tpm['thrL'].sort_values(ascending=False)

1046    5.454609e+10
1055    4.674812e+10
272     2.335403e+09
288     2.274023e+09
1474    8.278222e+08
            ...     
1812             NaN
1813             NaN
1838             NaN
1839             NaN
1840             NaN
Name: thrL, Length: 1864, dtype: float64

In [18]:
df_tpm.iloc[1046, :]

Unnamed: 0                          1046
dataset id    GENE_EXPRESSION_SRR4421267
thrL                  54546085099.831375
thrA                                 NaN
thrB                                 NaN
                         ...            
creC                                 NaN
creD                                 NaN
arcA                                 NaN
yjjY                                 NaN
yjtD                                 NaN
Name: 1046, Length: 4395, dtype: object

In [40]:
nulls_tpm = df_tpm.isnull().sum(axis=1)

In [41]:
for elem in nulls_tpm.sort_values(ascending=False).index:
    if nulls_tpm[elem] == 1000:
        break
    df_tpm.drop(index=elem, inplace=True)